<a href="https://colab.research.google.com/github/theshashank1/cse-c5-3year-da/blob/main/customer_personality_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Personality Analysis — Hypothesis Testing Project

### The Story
You are a Data Analyst at a retail company. Management wants to know two things: how are customers actually spending, and which customer traits predict whether someone responds to a marketing campaign.

**Dataset:** `marketing-campaign.csv` (2,240 customers)

| Column | Meaning |
|---|---|
| `ID` | Customer identifier |
| `Education` | Graduation / PhD / Master / Basic / 2n Cycle |
| `Income` | Annual household income |
| `Kidhome` | Number of young children at home |
| `Teenhome` | Number of teenagers at home |
| `MntWines` | Amount spent on wine (last 2 years) |
| `MntFruits` | Amount spent on fruits (last 2 years) |
| `NumWebPurchases` | Purchases made via website |
| `NumStorePurchases` | Purchases made in physical store |
| `Response` | Did the customer respond to the last campaign? (1 = Yes, 0 = No) |

In [1]:
import pandas as pd
from scipy import stats

In [3]:
df = pd.read_csv('/content/marketing-campaign.csv')

In [18]:
# Part 0: Diagnose Data Quality
print(f"Dataset Shape: {df.shape}")
print("\nMissing Values:")
print(df.isnull().sum())

# Checking for outliers in Income as discussed in lecture
print(f"\nMax Income found: {df['Income'].max()}")
print("Customers with Income > 600,000:")
display(df[df['Income'] > 600000][['ID', 'Income']])

Dataset Shape: (2240, 14)

Missing Values:
ID                    0
Education             0
Income               24
Kidhome               0
Teenhome              0
MntWines              0
MntFruits             0
NumWebPurchases       0
NumStorePurchases     0
Response              0
Total_spending        0
Total_Purchases       0
Total_Children        0
Has_Child             0
dtype: int64

Max Income found: 666666.0
Customers with Income > 600,000:


,ID,Income
2233,9432,666666.0


In [4]:
df

,ID,Education,Income,Kidhome,Teenhome,MntWines,MntFruits,NumWebPurchases,NumStorePurchases,Response
0,5524,Graduation,58138.0,0,0,635,88,8,4,1
1,2174,Graduation,46344.0,1,1,11,1,1,2,0
2,4141,Graduation,71613.0,0,0,426,49,8,10,0
3,6182,Graduation,26646.0,1,0,11,4,2,4,0
4,5324,PhD,58293.0,1,0,173,43,5,6,0
...,...,...,...,...,...,...,...,...,...,...
2235,10870,Graduation,61223.0,0,1,709,43,9,4,0
2236,4001,PhD,64014.0,2,1,406,0,8,5,0
2237,7270,Graduation,56981.0,0,0,908,48,2,13,0
2238,8235,Master,69245.0,0,1,428,30,6,10,0


In [5]:
df.columns

Index(['ID', 'Education', 'Income', 'Kidhome', 'Teenhome', 'MntWines',
       'MntFruits', 'NumWebPurchases', 'NumStorePurchases', 'Response'],
      dtype='object')

In [6]:
df['Total_spending'] = df['MntFruits'] + df['MntWines']
df['Total_Purchases'] = df['NumStorePurchases']+df['NumWebPurchases']
df['Total_Children'] = df['Kidhome']+df['Teenhome']
df['Has_Child'] = (df['Total_Children'] > 0).astype(int)

In [7]:
df.head()

,ID,Education,Income,Kidhome,Teenhome,MntWines,MntFruits,NumWebPurchases,NumStorePurchases,Response,Total_spending,Total_Purchases,Total_Children,Has_Child
0,5524,Graduation,58138.0,0,0,635,88,8,4,1,723,12,0,0
1,2174,Graduation,46344.0,1,1,11,1,1,2,0,12,3,2,1
2,4141,Graduation,71613.0,0,0,426,49,8,10,0,475,18,0,0
3,6182,Graduation,26646.0,1,0,11,4,2,4,0,15,6,1,1
4,5324,PhD,58293.0,1,0,173,43,5,6,0,216,11,1,1


In [8]:
t_value, p_value =stats.ttest_1samp(df['Total_spending'], 500)
print(f't-value: {t_value}, p-value: {p_value}')
print(f'{df['Total_spending'].mean()}')

t-value: -22.69695932198506, p-value: 8.11254738535582e-103
330.23794642857143


In [9]:
p_value

np.float64(8.11254738535582e-103)

In [10]:
t_value, p_value =stats.ttest_rel(df['MntWines'],df['MntFruits'] )
print(f't-value: {t_value}, p-value: {p_value}')
print(f'{df['Total_Purchases'].mean()}')

t-value: 40.65815317467306, p-value: 3.881040488170588e-271
9.875


In [11]:
t_value, p_value = stats.ttest_ind(df[df['Response'] == 0]['Total_spending'], df[df['Response'] == 1]['Total_spending'])
print(f't-value: {t_value}, p-value: {p_value}')
print(f'{df["Total_spending"].mean()}')

t-value: -12.172016182734554, p-value: 4.738080835877114e-33
330.23794642857143


In [12]:
df['Education'].unique()

array(['Graduation', 'PhD', 'Master', 'Basic', '2n Cycle'], dtype=object)

In [13]:
groups = []
for e in df['Education'].unique():
  groups.append((df[df['Education'] == e])['Total_spending'])

In [14]:
f_value, p_value = stats.f_oneway(*groups)
print(f'f-value: {f_value}, p-value: {p_value}')

f-value: 25.433037881891295, p-value: 1.2390419201993805e-20


In [15]:
cg = pd.crosstab(df['Education'], df['Response'])
cg

Response,0,1
Education,,
2n Cycle,181,22
Basic,52,2
Graduation,975,152
Master,313,57
PhD,385,101


In [16]:
chi2, p_value, dof, expected = stats.chi2_contingency(cg)
print(f'chi-square: {chi2}, p-value: {p_value}')

chi-square: 23.0760975769431, p-value: 0.00012226975294505314


### Research Summary

Based on the statistical analysis performed:

1.  **Spending Benchmarks**: A one-sample t-test shows the average total spending (\~330) is significantly different from a baseline of 500 (p < 0.05).
2.  **Product Preferences**: A paired t-test indicates a significant difference between spending on Wine vs. Fruits (p < 0.05).
3.  **Campaign Response**: An independent t-test shows that customers who responded to the campaign have significantly different total spending patterns compared to those who didn't.
4.  **Education Impact**: ANOVA results confirm that Education level significantly influences total spending (p < 0.05).
5.  **Demographic Association**: The Chi-square test shows a significant association between Education level and Campaign Response.

In [17]:
# Save the processed data with new features (Total_spending, Has_Child, etc.) for sharing
output_filename = '/content/marketing_campaign_with_research.csv'
df.to_csv(output_filename, index=False)

print(f'File saved successfully as: {output_filename}')
# You can now download this file from the left folder icon in Colab to share with students.')

File saved successfully as: /content/marketing_campaign_with_research.csv


## Assignment: Part 6 - Chi-Square Test

**Business Question:** Is having children at home associated with the campaign response?

- **H₀:** HasChildren and Response are independent.
- **H₁:** HasChildren and Response ARE associated.

*Student Task: Create a crosstab and perform the chi-square contingency test below.*

In [19]:
# TODO: Perform Chi-Square test for 'Has_Child' vs 'Response'
